## Описание проекта

Цель проекта — разработать алгоритм машинного обучения для предсказания целевых сессий пользователей.

Ключевые особенности:
1. **Сильный дисбаланс классов:** 
2. **Специфика целевой метрики:** 
3. **Временная структура логов:** 


## Мой подход к решению

Чтобы получить высокую точность ранжирования и адаптировать модель под метрику Precision@Recall >= 0.70, я спроектировала следующий пайплайн:

### 1. Фильтрация событий по интервалу
Из общего массива событий отбираются только те действия, которые попадают в окно наблюдения для каждого конкретного. Это отсекает фоновый шум и изолирует активность сессии.

### 2. Инжиниринг поведенческих признаков 
Основной упор сделан на поиск аномалий и выявление паттернов бот-активности:
* **Анализ временных интервалов:**
* **Анализ курсора:**
* **Разнообразие сессии:**
* **Распределение событий и платформ:**

### 3. Моделирование 

* **Выбор алгоритма:** **`HistGradientBoostingClassifier`** благодаря быстрой работе на гистограммах , встроенной обработке пропусков `NaN` и нативной поддержке аргумента `class_weight='balanced'` для борьбы с дисбалансом.

In [34]:
# Загружаем библиотеки

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from metric import precision_at_recall

In [35]:
RANDOM_STATE = 42

# Читаем данные
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(f"Загружено: Train {train.shape}, Test {test.shape}, Events {events.shape}")

Загружено: Train (11091, 5), Test (4909, 4), Events (328905, 14)


In [37]:
# Фильтрация событий по окну

def get_events_in_window(events_df, meta_df):
    """Фильтрация событий по интервалу наблюдения."""
    ev = events_df.merge(meta_df[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', how='inner')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = get_events_in_window(events, train)
ev_te = get_events_in_window(events, test)

print(f"События в окне: Train = {len(ev_tr)}, Test = {len(ev_te)}")

События в окне: Train = 198436, Test = 89690


In [38]:
# Генерация признаков

def extract_advanced_features(ev_df, meta_df):
    """Извлечение поведенческих признаков."""
    ev = ev_df.sort_values(['cookie_id', 'event_ts'])
    
    # Паузы между последовательными кликами
    ev['time_diff'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
    
    g = ev.groupby('cookie_id')
    
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g['item_id'].nunique(),
        'cat_nunique': g['item_category'].nunique(),
        'loc_nunique': g['item_location'].nunique(),
        'event_types_nunique': g['event_name'].nunique(),
        'platform_nunique': g['platform'].nunique(),
        'ua_nunique': g['user_agent'].nunique(),
        'seller_type_nunique': g['seller_type'].nunique(),
        'duration_sec': (g['event_ts'].max() - g['event_ts'].min()).dt.total_seconds(),
        
        # Анализ таймингов кликов
        'min_time_diff': g['time_diff'].min(),
        'mean_time_diff': g['time_diff'].mean(),
        'median_time_diff': g['time_diff'].median(),
        'std_time_diff': g['time_diff'].std(),
        'zero_diff_count': (ev['time_diff'] == 0).groupby(ev['cookie_id']).sum(),
        'fast_events_count': (ev['time_diff'] < 0.5).groupby(ev['cookie_id']).sum(),
        
        # Координаты pointer_x, pointer_y
        'pointer_x_std': g['pointer_x'].std(),
        'pointer_y_std': g['pointer_y'].std(),
        'pointer_x_nunique': g['pointer_x'].nunique(),
        'pointer_y_nunique': g['pointer_y'].nunique(),
    })
    
    piv_event = ev.pivot_table(index='cookie_id', columns='event_name', values='event_ts', aggfunc='count', fill_value=0)
    piv_event.columns = [f'event_{c}' for c in piv_event.columns]
    
    piv_platform = ev.pivot_table(index='cookie_id', columns='platform', values='event_ts', aggfunc='count', fill_value=0)
    piv_platform.columns = [f'platform_{c}' for c in piv_platform.columns]
    
    res = meta_df[['cookie_id', 'cookie_created_at', 'window_start_ts']].merge(f.reset_index(), on='cookie_id', how='left')
    res = res.merge(piv_event.reset_index(), on='cookie_id', how='left')
    res = res.merge(piv_platform.reset_index(), on='cookie_id', how='left')
    
    res['cookie_age_sec'] = (res['window_start_ts'] - res['cookie_created_at']).dt.total_seconds()
    res['events_per_sec'] = res['n_events'] / (res['duration_sec'] + 1)
    res['items_per_event'] = res['item_nunique'] / (res['n_events'] + 1e-5)
    
    return res.fillna(0)

Xtr_adv = extract_advanced_features(ev_tr, train)
Xte_adv = extract_advanced_features(ev_te, test)

drop_cols = ['cookie_id', 'cookie_created_at', 'window_start_ts']
feats = [c for c in Xtr_adv.columns if c not in drop_cols]

Xtr = Xtr_adv[feats]
ytr = train['target'].values
Xte = Xte_adv[feats]

print(f"Сформировано признаков: {len(feats)}")

Сформировано признаков: 42


In [40]:
# Валидация по времени

is_valid = train['window_start_ts'].ge('2026-04-17').values

model_val = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_leaf_nodes=31,
    max_iter=400,
    l2_regularization=2.0,
    class_weight='balanced',
    random_state=RANDOM_STATE
)

model_val.fit(Xtr.loc[~is_valid], ytr[~is_valid])
val_preds = model_val.predict_proba(Xtr.loc[is_valid])[:, 1]

val_score = precision_at_recall(ytr[is_valid], val_preds)
print(f"Локальная метрика Precision@Recall>=0.70: {val_score:.5f}")

Локальная метрика Precision@Recall>=0.70: 0.75676


In [41]:
# Обучение на полном train
final_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_leaf_nodes=31,
    max_iter=400,
    l2_regularization=2.0,
    class_weight='balanced',
    random_state=RANDOM_STATE
)

final_model.fit(Xtr, ytr)
test_preds = final_model.predict_proba(Xte)[:, 1]

# Чтение образца 
sample_sub = pd.read_csv('sample_submission.csv')
raw_sub = pd.DataFrame({'cookie_id': Xte_adv['cookie_id'], 'score': test_preds})

submission = sample_sub[['cookie_id']].merge(raw_sub, on='cookie_id', how='left')

# Проверки целостности
assert len(submission) == len(sample_sub), "Число строк не совпадает!"
assert not submission['score'].isna().any(), "Есть пропущенные значения NaN!"

submission.to_csv('submission.csv', index=False)
print("Итоговый submission.csv успешно сгенерирован и сохранен!")
submission.head()

Итоговый submission.csv успешно сгенерирован и сохранен!


,cookie_id,score
0,ck_99a4e5ef02d89493,0.239612
1,ck_54d463f7fd89ec3d,0.426484
2,ck_0fbbc7a6af300368,0.036200
3,ck_8fd4937eadd64fb6,0.029720
4,ck_dbb4bd4eda97255d,0.084182




# Итоговые результаты 

## 1. Результаты работы
В рамках проекта я реализовала полный пайплайн: от очистки сырых логов и генерации 42 поведенческих признаков до настройки временной валидации, обучения модели и формирования итогового файла

* **Константный базовый уровень платформы:** `0.09391`
* **Локальная валидация :** **`0.75676`** (Precision@Recall >= 0.70)
* **Результат на тестовом сервере:** **`0.70217`**



## 2. Выводы

1. **Эффективность поведенческих фичей:** 
   Улучшили качество паузы между кликами, доля быстрых событий и разброс координат курсора. Это позволило увидеть различия между реальным пользователем и автоматическими запросами.
2. **Оконная фильтрация:** 
   Ограничение событий интервалом помогло полностью очистить агрегаты от  шума прошлых дней.
3. **Выбор HistGradientBoostingClassifier:** 
   Использование биннинга признаков и балансировки весов классов при умеренной L2-регуляризации позволило получить сглаженные и хорошо отранжированные вероятности без переобучения.



## 3. Проверка артефактов
* Все проверки целостности пройдены: отсутствуют пропуски, порядок и количество строго соответствуют sample_submission.csv.